# Lab 11：WGAN-GP 產生人臉（Colab 版）

**Colab 版說明**（張珀銀 2026/10/16）

課本程式寫於 TensorFlow 2.0 時代，在現在的 Colab 會出錯。這一版只改必要的地方，改過的儲存格開頭都標了「【Colab 版修改】」，其餘和課本相同。改了：
- 課本 Lab11 是外部專案 [TF2-WGAN](https://github.com/KUASWoodyLIN/TF2-WGAN)（作者 Woody Lin，MIT 授權），原本要自己 clone 再照 README 執行。這一本把步驟寫好，在 Colab 裡 clone、修正、訓練
- 第一格「Colab 準備」：舊版 Keras 相容設定
- 人臉資料 celeb_a 改讀老師提供的 `celeba_15k.zip`（CelebA 前 15,000 張對齊照片）。原本 tfds 從 Google 雲端硬碟下載，檔案太大會先跳出掃毒確認頁，下載必定失敗
- `train.py` 的 `CUDA_VISIBLE_DEVICES` 由 `"1"` 改 `"0"`：Colab 只有一張 GPU，編號是 0，寫 1 會找不到 GPU

**開始前**：
1. 在「執行階段 → 變更執行階段類型」選 GPU。要先做這一步，換執行階段類型會清掉已上傳的檔案。
2. 下載老師提供的 `celeba_15k.zip`。CelebA 只授權非商業研究使用，這個檔案只供本課使用，請勿轉傳或放上網路。在 Colab 左側點資料夾圖示，按「上傳」選 `celeba_15k.zip`，檔案會放在 Colab 預設資料夾 `/content`。等上傳進度圈跑完再執行。
3. 選「執行階段 → 全部執行」。

「執行階段 → 重新啟動工作階段」不會清掉檔案，重新啟動後從第一格開始執行即可；Colab 斷線或換執行階段類型時檔案會被清掉，要再上傳一次。

In [ ]:
# 【Colab 版修改】Colab 準備：每次開啟或重新啟動後，最先執行這一格
import os
import zipfile
DATA_FILE = 'celeba_15k.zip'  # 上傳到 Colab 預設資料夾 /content 的檔案
# 課本寫於 TF 2.0 時代，改用 Colab 內建的舊版 Keras（tf_keras）；下面用 !python 執行的程式也沿用這個設定
os.environ['TF_USE_LEGACY_KERAS'] = '1'

# 下載課本的專案並進入資料夾（重複執行這一格時不會再做一次）
if os.path.basename(os.getcwd()) != 'TF2-WGAN':
    if not os.path.exists(DATA_FILE):
        raise FileNotFoundError('找不到 celeba_15k.zip：請在左側資料夾按「上傳」放入老師提供的 celeba_15k.zip，等上傳完成再執行這一格')
    if not os.path.exists('TF2-WGAN'):
        !git clone -q https://github.com/KUASWoodyLIN/TF2-WGAN
    zipfile.ZipFile(DATA_FILE).extractall('TF2-WGAN')  # 解壓成 TF2-WGAN/celeba/ 資料夾
    %cd TF2-WGAN

### 新增讀照片的函數：做成和 tfds 一樣的格式，`utils/dataset.py` 的 `parse_fn` 不用改

In [ ]:
%%writefile utils/celeba_files.py
# 【Colab 版新增】celeb_a 改讀老師提供的照片：tfds 從 Google 雲端硬碟下載會失敗
import tensorflow as tf


def load_celeba(folder):
    files = tf.data.Dataset.list_files(folder + '/*.jpg', shuffle=False)

    def read(path):
        img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        img.set_shape((218, 178, 3))  # CelebA 對齊照片固定 218x178，parse_fn 裁切時要用到這個大小
        return {'image': img}  # 和 tfds 一樣的 {'image': ...} 格式

    return files.map(read, num_parallel_calls=tf.data.experimental.AUTOTUNE)


### 修改 `train.py` 兩處

In [ ]:
# 【Colab 版修改】專案程式只改兩處，其餘和課本相同
def fix(path, old, new):
    # 把 path 檔案裡的 old 換成 new；已經換過就跳過，內容和預期不同就停下來
    text = open(path, encoding='utf-8').read()
    if old in text:
        open(path, 'w', encoding='utf-8').write(text.replace(old, new))
    elif new not in text:
        raise RuntimeError(path + ' 的內容和預期不同，請告訴老師')

# Colab 只有一張 GPU，編號是 0；原本寫 1 會找不到 GPU，改用 CPU 會慢很多
fix('train.py', 'os.environ["CUDA_VISIBLE_DEVICES"] = "1"', 'os.environ["CUDA_VISIBLE_DEVICES"] = "0"')
# celeb_a 改讀解壓好的 celeba/ 資料夾
fix('train.py', "train_data, info = tfds.load(dataset, split='train+validation+test', with_info=True)",
    "from utils.celeba_files import load_celeba\ntrain_data = load_celeba('celeba')")

### 確認用的是舊版 Keras（應顯示 tf_keras）

In [ ]:
!python -c "import tensorflow as tf; print('TensorFlow', tf.__version__, '；Keras 用的是', tf.keras.__name__.split('.')[0])"

### 訓練：`train.py`（25 個 epoch；生成器每訓練 100 次，就把 100 張產生的人臉存到 TensorBoard）

In [ ]:
!python train.py

### TensorBoard：看損失曲線，並在 IMAGES 分頁看人臉隨訓練逐漸成形

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs_wgan_2